# Part 1 - Forecast the target (125 m wind, t+1 / t+7 / t+14)

Predict wind speed (`q05/q50/q95`) and direction (`dir_05/dir_50/dir_95`) at 125 m, for three lead times. Produces the coarse forecast; `2_downscale_to_target` turns it into the submission.

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
#   os.environ['PHASE2_DATA_ROOT'] = '/path/to/unzipped/datasets'
import config; print(config.describe())
import pickle, numpy as np, pandas as pd
import forecast_pipeline as P
import splits

## 1. Train the forecast

In [ ]:
train = P.train_dates('6D')          # '3D' for a stronger (slower) fit
mos, qmos, adj, offs = P.fit_forecast(train)
print('conformal width per lead:', {k: round(v,2) for k,v in adj.items()})
print('direction half-width (deg) per horizon:', {k: round(v,1) for k,v in offs.items()})

## 2. Forecast every evaluation window

In [ ]:
windows = splits.eval_windows()
cache = {'offs': offs, 'models': (mos, qmos, adj)}
for wi, w in enumerate(windows):
    cache[wi] = P.coarse_fields(mos, qmos, adj, P.issue_date_of(w))
    print(f'window {wi}: issue {P.issue_date_of(w).date()}  ({len(cache[wi])} fields)')
Path = __import__('pathlib').Path
Path('part1_forecast/cache').mkdir(exist_ok=True)
with open('part1_forecast/cache/coarse_forecasts.pkl', 'wb') as f:
    pickle.dump(cache, f)
print('cached', len(windows), 'windows ->', 'part1_forecast/cache/coarse_forecasts.pkl')